In [5]:
import torch
from torch import nn
from d2l import torch as d2l

In [6]:
# Sequence-To-Sequence Parameter Initialization

def init_seq2seq(
    module: nn.Module,
) -> None:
    
    if type(module) is nn.Linear:
        nn.init.xavier_uniform_(
            module.weight
        )

    if type(module) is nn.GRU:        
        for name, parameter in module.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(
                    parameter
                )


# Sequence-to-Sequence Encoder

class Seq2SeqEncoder(d2l.Encoder):
    
    def __init__(
        self,
        vocab_size: int,
        embed_size: int,
        num_hiddens: int,
        num_layers: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_size,
        )
        
        self.rnn = d2l.GRU(
            num_inputs=embed_size,
            num_hiddens=num_hiddens,
            num_layers=num_layers,
            dropout=dropout,
        )
        
        self.apply(
            init_seq2seq
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
        *args: object,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,  
    ]:
        
        # [nn.Embedding]:
        # [B, T] -> [T, B] -> [T, B, E]
        embeddings = self.embedding(
            X.T.to(dtype=torch.long)
        )
        
        # [nn.GRU]: 
        # inputs : [T, B, E]
        # outputs: [T, B, H]
        # state  : [L, B, H]
        outputs, state = self.rnn(
            embeddings 
        )
        
        return outputs, state


# Encoder tensor shape 체크

vocab_size = 10
embed_size = 8
num_hiddens = 16
num_layers = 2
batch_size = 4
num_steps = 9

encoder = Seq2SeqEncoder(
    vocab_size=vocab_size,
    embed_size=embed_size,
    num_hiddens=num_hiddens,
    num_layers=num_layers,
)

# [B, T] = [4, 9]
X = torch.zeros(
    (batch_size, num_steps),
    dtype=torch.long,
)

with torch.no_grad():
    encoder_outputs, encoder_state = encoder(X)

print(
    "Input shape:",
    tuple(X.shape),
)
print(
    "Encoder outputs shape:",
    tuple(encoder_outputs.shape),
)
print(
    "Encoder state shape:",
    tuple(encoder_state.shape),
)

Input shape: (4, 9)
Encoder outputs shape: (9, 4, 16)
Encoder state shape: (2, 4, 16)


In [7]:
# Sequence-To-Sequence Decoder

class Seq2SeqDecoder(d2l.Decoder):
    
    def __init__(
        self,
        vocab_size: int,
        embed_size: int,
        num_hiddens: int,
        num_layers: int,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_size,
        )
        
        self.rnn = d2l.GRU(
            num_inputs=embed_size + num_hiddens,
            num_hiddens=num_hiddens,
            num_layers=num_layers,
            dropout=dropout,
        )
        
        self.dense = nn.LazyLinear(
            out_features=vocab_size
        )
        
        self.apply(
            init_seq2seq
        )
        
        
    def init_state(
        self,
        enc_all_outputs: tuple[
            torch.Tensor,
            torch.Tensor,
        ],
        *args: object,
    ) -> tuple[
        torch.Tensor,
        torch.Tensor,
    ]:
        return enc_all_outputs
    
    
    def forward(
        self,
        X: torch.Tensor,
        state: tuple[
            torch.Tensor,
            torch.Tensor,
        ],
    ) -> tuple[
        torch.Tensor,
        tuple[torch.Tensor, torch.Tensor],
    ]:
    
        # Decoder input X: [B, T]
        # [B, T] -> [T, B] -> [T, B, E]
        embeddings = self.embedding(
            X.T.to(dtype=torch.long)
        )

        # encoder_outputs: [T, B, H]
        # hidden_state   : [L, B, H]
        encoder_outputs, hidden_state = state

        # Encoder의 마지막 time step 선택
        # [T, B, H] -> [B, H]
        context = encoder_outputs[-1]

        # time step T개로 복사
        # [B, H] -> [T, B, H]
        context = context.repeat(
            embeddings.shape[0], # T
            1, 
            1,
        )
        
        # Feature dimension으로 Concatenate:
        # [T, B, E] + [T, B, H] -> [T, B, E + H]
        embeddings_and_context = torch.cat(
            (embeddings, context),
            dim=-1,
        )
        
        # Forward into GRU
        # outputs      : [T, B, H]
        # hidden_state : [L, B, H]
        outputs, hidden_state = self.rnn(
            embeddings_and_context,
            hidden_state,
        )
        
        # Linear: H → V & 축 교환
        # [T, B, H] -> [T, B, V] -> [B, T, V]
        outputs = self.dense(
            outputs
        ).swapaxes(0, 1)
        
        
        return outputs, (
            encoder_outputs,
            hidden_state,
        )

In [8]:
# Decoder Tensor Shape Check

decoder = Seq2SeqDecoder(
    vocab_size=vocab_size,
    embed_size=embed_size,
    num_hiddens=num_hiddens,
    num_layers=num_layers,
)

state = decoder.init_state(
    encoder(X)
)

with torch.no_grad():
    decoder_outputs, state = decoder(
        X,
        state,
    )

print(
    "Input shape:",
    tuple(X.shape),
)
print(
    "Encoder outputs shape:",
    tuple(encoder_outputs.shape),
)
print(
    "Encoder state shape:",
    tuple(encoder_state.shape),
)

print(
    "Decoder outputs shape:",
    tuple(decoder_outputs.shape),
)
print(
    "Decoder hidden state shape:",
    tuple(state[1].shape),
)

d2l.check_shape(
    decoder_outputs,
    (batch_size, num_steps, vocab_size),
)
d2l.check_shape(
    state[1],
    (num_layers, batch_size, num_hiddens),
)

Input shape: (4, 9)
Encoder outputs shape: (9, 4, 16)
Encoder state shape: (2, 4, 16)
Decoder outputs shape: (4, 9, 10)
Decoder hidden state shape: (2, 4, 16)
